In [654]:
import pandas as pd
import re
import unicodedata
import os
import duckdb

#### Leitura e transformação dos dados

In [655]:
# Definindo a estrutura de caminhos relativos do projeto
BRONZE_DIR = "../data/bronze"
SILVER_DIR = "../data/silver"

In [656]:
def ler_nomes_arquivos_bronze():
    """
    Lê os nomes dos arquivos na pasta bronze e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(BRONZE_DIR) if os.path.isfile(os.path.join(BRONZE_DIR, f))]

In [657]:
lista_arquivos = ler_nomes_arquivos_bronze()

In [658]:
def transformar_csv_para_parquet(lista_arquivos):
    """
    Função para transformar um arquivo CSV em Parquet.
    
    Parâmetros:
    lista_arquivos (list): Lista de nomes dos arquivos CSV de entrada.
    """

    for arquivo in lista_arquivos:
        # Lendo o arquivo CSV
        df = pd.read_csv(os.path.join(BRONZE_DIR, arquivo))
        
        # Definindo o nome do arquivo Parquet de saída
        nome_arquivo_parquet = os.path.splitext(arquivo)[0] + '.parquet'
        
        # Salvando o DataFrame como Parquet
        df.to_parquet(os.path.join(SILVER_DIR, nome_arquivo_parquet), index=False)
        
        print(f"Arquivo {arquivo} transformado")


In [659]:
transformar_csv_para_parquet(lista_arquivos)

Arquivo br_bd_diretorios_brasil_cnae_2.csv transformado
Arquivo br_bd_diretorios_brasil_municipio.csv transformado
Arquivo br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv transformado


In [660]:
def ler_nomes_arquivos_silver():
    """
    Lê os nomes dos arquivos na pasta silver e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(SILVER_DIR) if os.path.isfile(os.path.join(SILVER_DIR, f))]

In [661]:
ler_nomes_arquivos_silver()

['br_bd_diretorios_brasil_cnae_2.parquet',
 'br_bd_diretorios_brasil_municipio.parquet',
 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet']

In [662]:
operacoes = pd.read_parquet(os.path.join(SILVER_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet'))

operacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 23483 entries, 0 to 23482
Data columns (total 39 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   razao_social_cliente                     23483 non-null  str    
 1   cnpj_cliente                             23474 non-null  float64
 2   descricao_projeto                        23483 non-null  str    
 3   sigla_uf                                 23483 non-null  str    
 4   nome_municipio                           23483 non-null  str    
 5   id_municipio                             15762 non-null  float64
 6   id_contrato                              23483 non-null  int64  
 7   data_contratacao                         23483 non-null  str    
 8   valor_contratado                         23483 non-null  float64
 9   valor_desembolsado                       23483 non-null  float64
 10  tipo_fonte_recursos                      22647 non-null  

#### Tratamento de valores nulos

In [663]:
def verificar_valores_nulos(df):
    """Verifica se um DataFrame contém valores nulos.

    Parâmetros:
        df (DataFrame): O DataFrame a ser verificado.

    Retorna:
        False | DataFrame: False se não houver nulos; DataFrame com análise se houver.
    """
    total_nulos = df.isnull().sum().sort_values(ascending=False)
    total_nulos = total_nulos[total_nulos > 0]

    if total_nulos.empty:
        return False

    total_nulos_percent = ((total_nulos / df.shape[0]) * 100).round(2)
    return pd.DataFrame({'Total Nulls': total_nulos, '%': total_nulos_percent})

In [664]:
verificar_valores_nulos(operacoes)

,Total Nulls,%
tipo_excepcionalidade,23265,99.07
cnpj_instituicao_financeira_credenciada,19412,82.66
nome_instituicao_financeira_credenciada,19412,82.66
id_municipio,7721,32.88
subclasse_cnae,5507,23.45
tipo_fonte_recursos,836,3.56
classe_cnae,497,2.12
situacao_contrato,268,1.14
grupo_cnae,255,1.09
cnpj_cliente,9,0.04


In [665]:
#categorias existentes na coluna tipo_excepcionalidade
operacoes["tipo_excepcionalidade"].value_counts(dropna=False)

tipo_excepcionalidade
NaN                                                                                  23265
CONDIÇÕES DE CRÉDITO                                                                    81
CONDIÇÕES FINANCEIRAS E OPERACIONAIS                                                    68
COBRANÇA DE COMISSÕES                                                                   47
CONDIÇÕES FINANCEIRAS E OPERACIONAIS /CONDIÇÕES DE CRÉDITO                              17
GARANTIAS                                                                                3
CONDIÇÕES FINANCEIRAS E OPERACIONAIS /COBRANÇA DE COMISSÕES /CONDIÇÕES DE CRÉDITO        2
Name: count, dtype: int64

**Regra de negócio:** Se _tipo_excepcionalidade_
 está nulo, significa que a operação seguiu o fluxo padrão (sem exceção). Então, podemos realizar a classificação 0 = não, e 1 = sim.

In [666]:
#Criando uma nova coluna para indicar se a operação possui ou não excepcionalidade
operacoes["tem_excepcionalidade"] = operacoes["tipo_excepcionalidade"].notna().astype(int)

In [667]:
operacoes = operacoes.drop(columns=["tipo_excepcionalidade"])

In [668]:
operacoes.isnull().sum().sort_values(ascending=False).head(10)

nome_instituicao_financeira_credenciada    19412
cnpj_instituicao_financeira_credenciada    19412
id_municipio                                7721
subclasse_cnae                              5507
tipo_fonte_recursos                          836
classe_cnae                                  497
situacao_contrato                            268
grupo_cnae                                   255
cnpj_cliente                                   9
id_contrato                                    0
dtype: int64

**Regra de Négocio:** Se o CNPJ da instituição financeira credenciada estiver nulo, podemos entender que a operação foi realizada diretamente com o BNDES, sem intermediação de uma instituição financeira. Portanto, vamos preencher esses valores nulos com a string "OPERAÇÃO DIRETA".

In [669]:
operacoes["cnpj_instituicao_financeira_credenciada"] = operacoes["cnpj_instituicao_financeira_credenciada"].fillna("0000000000000.0")
operacoes["nome_instituicao_financeira_credenciada"] = operacoes["nome_instituicao_financeira_credenciada"].fillna("OPERAÇÃO DIRETA")
print(f"{operacoes[['cnpj_instituicao_financeira_credenciada']].dtypes}")

cnpj_instituicao_financeira_credenciada    object
dtype: object


In [670]:
#Verficando colunas restantes para tratamento de valores nulos
operacoes.isnull().sum().sort_values(ascending=False).head(10)

id_municipio           7721
subclasse_cnae         5507
tipo_fonte_recursos     836
classe_cnae             497
situacao_contrato       268
grupo_cnae              255
cnpj_cliente              9
id_contrato               0
nome_municipio            0
sigla_uf                  0
dtype: int64

In [671]:
print(f"{operacoes[['cnpj_cliente']].dtypes}")

cnpj_cliente    float64
dtype: object


In [672]:
#preenchendo os valores nulos
operacoes["id_municipio"] = operacoes["id_municipio"].fillna("NÃO INFORMADO")
operacoes["cnpj_cliente"] = operacoes["cnpj_cliente"].fillna("000000000000.0")
operacoes["situacao_contrato"] = operacoes["situacao_contrato"].fillna("OUTROS")
operacoes["tipo_fonte_recursos"] = operacoes["tipo_fonte_recursos"].fillna("OUTROS")

In [673]:
#removendo colunas cnae desnecessárias
operacoes = operacoes.drop(columns=["classe_cnae","subclasse_cnae","grupo_cnae","divisao_cnae","secao_cnae"])

In [674]:
verificar_valores_nulos(operacoes)

False

#### Validação dos tipos de dados e Enriquecimento

In [675]:
#Verificando os tipos de dados das colunas
operacoes.dtypes

razao_social_cliente                           str
cnpj_cliente                                object
descricao_projeto                              str
sigla_uf                                       str
nome_municipio                                 str
id_municipio                                object
id_contrato                                  int64
data_contratacao                               str
valor_contratado                           float64
valor_desembolsado                         float64
tipo_fonte_recursos                            str
custo_financeiro                               str
taxa_juros                                 float64
prazo_carencia                               int64
prazo_amortizacao                            int64
modalidade_apoio                               str
forma_apoio                                    str
produto                                        str
tipo_instrumento_financeiro                    str
indicador_inovacao             

In [676]:
#consultando colunas de data
cols_data = [c for c in operacoes.columns if "data" in c]
cols_data
operacoes[cols_data].dtypes

data_contratacao    str
data_apuracao       str
dtype: object

In [677]:
#Ajuste dos tipos de dados das colunas de data
operacoes[cols_data] = operacoes[cols_data].apply(pd.to_datetime, errors='coerce')
operacoes[cols_data].dtypes

data_contratacao    datetime64[us]
data_apuracao       datetime64[us]
dtype: object

In [678]:
operacoes["ano_contratacao"] = operacoes["data_contratacao"].dt.year
operacoes["mes_contratacao"] = operacoes["data_contratacao"].dt.month

### Limpeza e padronização de textos

In [685]:
def normalize_text(value):
    if pd.isna(value):
        return value
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"\s+", " ", text)
    return text

In [686]:
def padronizar_texto(df, cols=None):
    df_limpo = df.copy()
    if cols is None:
        cols = df_limpo.select_dtypes(include=["object", "string"]).columns
    for col in cols:
        df_limpo[col] = df_limpo[col].map(normalize_text)
    return df_limpo

In [687]:
operacoes = padronizar_texto(operacoes)

In [689]:
silver_operacoes = operacoes.copy()

total = silver_operacoes[["valor_contratado", "valor_desembolsado"]].sum().round(2)
total_silver= total.apply(lambda x: f"{x:,.2f}")
total_silver

valor_contratado      1,229,184,483,691.93
valor_desembolsado      952,046,450,761.74
dtype: str

In [690]:
# Ler os dados brutos
raw = pd.read_csv(
    os.path.join(BRONZE_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv'),
    dtype=str
)

cols = ["valor_contratado", "valor_desembolsado"]

In [691]:
total = raw[cols].apply(pd.to_numeric, errors="coerce").sum().round(2)

total__rawformatado = total.apply(lambda x: f"{x:,.2f}")
total_raw_formatado

valor_contratado      1,229,184,483,691.93
valor_desembolsado      952,046,450,761.74
dtype: str